In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 94.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=094cd120ffcbba87768a883acb7d58aa483d8431b90b1d3381368a7efda8cd99
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

def sim_bb84_attacker(num_bits):
  # Ensure we don't exceed the 24-qubit limit of BasicSimulator
  if num_bits > 24:
    print("Reducing bits to 24 due to BasicSimulator limits.")
    num_bits = 24

  alice_bits = [random.randint(0, 1) for _ in range(num_bits)]
  alice_bases = [random.randint(0, 1) for _ in range(num_bits)]

  qc = QuantumCircuit(num_bits, num_bits)

  # Alice prepares qubit sequence
  for i in range(num_bits):
        if alice_bits[i] == 1: qc.x(i)
        if alice_bases[i] == 1: qc.h(i)

  # Eve Intercepts (The "Intercept-Resend" attack)
  # Eve measures every qubit in a random basis
  eve_bases = [random.randint(0, 1) for _ in range(num_bits)]
  for i in range(num_bits):
    if eve_bases[i] == 1: qc.h(i)
    qc.measure(i, i) # Eve's measurement
    if eve_bases[i] == 1: qc.h(i) # Re-prepares for Bob

  # Bob receives and measures
  bob_bases = [random.randint(0, 1) for _ in range(num_bits)]
  for i in range(num_bits):
    if bob_bases[i] == 1: qc.h(i)
    qc.measure(i, i)

  # Run simulation
  backend = BasicSimulator()
  job = backend.run(transpile(qc, backend), shots=1)
  measured_bits = list(job.result().get_counts().keys())[0][::-1]
  bob_results = [int(b) for b in measured_bits]

  # Sifting and Error Check
  sifted_alice = []
  sifted_bob = []
  for i in range(num_bits):
    if alice_bases[i] == bob_bases[i]:
      sifted_alice.append(alice_bits[i])
      sifted_bob.append(bob_results[i])

  # Any attack will mean Alice’s and Bob’s bit strings, in the positions in which they used the same basis, will sometimes not agree.
  errors = sum(1 for a, b in zip(sifted_alice, sifted_bob) if a != b)
  error_rate = errors / len(sifted_alice) if sifted_alice else 0

  print(f"Error rate detected: {error_rate:.2%}")
  if error_rate > 0.20: # Theoretically ~25% with Eve[cite: 1]
    print("ALERT: Eavesdropper detected! Protocol aborted.")
  else:
    print("Channel secure.")

# Alice and Bob can detect the attack by comparing a subset of their bit strings, to check they really do agree.
# This comparison is done publicly, which means that those bits should not be used as part of the final key.
# By sacrificing enough bits, the probability of detecting the attack can be made arbitrarily high.
sim_bb84_attacker(100)



Reducing bits to 24 due to BasicSimulator limits.
Error rate detected: 33.33%
ALERT: Eavesdropper detected! Protocol aborted.
